In [1]:
import sys
sys.path.insert(0, '../..')

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

PROC = '../../data/processed/'
FEAT = '../../data/features/'

print("✅ Imports ready")

✅ Imports ready


In [2]:
from src.utils.config import settings

# Build connection string
DB_URL = (
    f"postgresql://{settings.POSTGRES_USER}"
    f":{settings.POSTGRES_PASSWORD}"
    f"@{settings.POSTGRES_HOST}"
    f":{settings.POSTGRES_PORT}"
    f"/{settings.POSTGRES_DB}"
)

engine = create_engine(DB_URL, echo=False)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT version()"))
    version = result.fetchone()[0]
    print(f"✅ Connected to PostgreSQL")
    print(f"   Version: {version[:50]}")

✅ Connected to PostgreSQL
   Version: PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on aarc


In [3]:
## Create Star Schema Tables

schema_sql = """
-- ─────────────────────────────────────────────
-- Drop tables if they exist (clean slate)
-- ─────────────────────────────────────────────
DROP TABLE IF EXISTS fact_ratings    CASCADE;
DROP TABLE IF EXISTS dim_movies      CASCADE;
DROP TABLE IF EXISTS dim_users       CASCADE;
DROP TABLE IF EXISTS dim_time        CASCADE;
DROP TABLE IF EXISTS dim_genres      CASCADE;
DROP TABLE IF EXISTS bridge_movie_genre CASCADE;

-- ─────────────────────────────────────────────
-- Dimension: Movies
-- ─────────────────────────────────────────────
CREATE TABLE dim_movies (
    movie_id         INTEGER PRIMARY KEY,
    tmdb_id          INTEGER,
    imdb_id          VARCHAR(20),
    title            TEXT,
    original_title   TEXT,
    overview         TEXT,
    tagline          TEXT,
    original_language VARCHAR(10),
    release_year     INTEGER,
    era              VARCHAR(20),
    runtime          FLOAT,
    budget           FLOAT,
    revenue          FLOAT,
    vote_average     FLOAT,
    vote_count       FLOAT,
    popularity       FLOAT,
    director         TEXT,
    top_cast         TEXT,
    poster_path      TEXT,
    rating_count     INTEGER,
    rating_mean      FLOAT,
    popularity_tier  VARCHAR(10)
);

-- ─────────────────────────────────────────────
-- Dimension: Users
-- ─────────────────────────────────────────────
CREATE TABLE dim_users (
    user_id          INTEGER PRIMARY KEY,
    rating_count     INTEGER,
    rating_mean      FLOAT,
    rating_std       FLOAT,
    unique_movies    INTEGER,
    activity_tier    VARCHAR(10),
    rating_tendency  VARCHAR(10),
    top_genre        VARCHAR(50)
);

-- ─────────────────────────────────────────────
-- Dimension: Time
-- ─────────────────────────────────────────────
CREATE TABLE dim_time (
    time_id          SERIAL PRIMARY KEY,
    timestamp        TIMESTAMP,
    year             INTEGER,
    month            INTEGER,
    day              INTEGER,
    hour             INTEGER,
    day_of_week      INTEGER,
    quarter          INTEGER
);

-- ─────────────────────────────────────────────
-- Dimension: Genres
-- ─────────────────────────────────────────────
CREATE TABLE dim_genres (
    genre_id         SERIAL PRIMARY KEY,
    genre_name       VARCHAR(50) UNIQUE
);

-- ─────────────────────────────────────────────
-- Bridge: Movie ↔ Genre (many to many)
-- ─────────────────────────────────────────────
CREATE TABLE bridge_movie_genre (
    movie_id         INTEGER REFERENCES dim_movies(movie_id),
    genre_id         INTEGER REFERENCES dim_genres(genre_id),
    PRIMARY KEY (movie_id, genre_id)
);

-- ─────────────────────────────────────────────
-- Fact: Ratings (central fact table)
-- ─────────────────────────────────────────────
CREATE TABLE fact_ratings (
    rating_id        SERIAL PRIMARY KEY,
    user_id          INTEGER REFERENCES dim_users(user_id),
    movie_id         INTEGER REFERENCES dim_movies(movie_id),
    time_id          INTEGER REFERENCES dim_time(time_id),
    rating           FLOAT,
    timestamp        TIMESTAMP
);

-- Indexes for fast joins
CREATE INDEX idx_fact_user    ON fact_ratings(user_id);
CREATE INDEX idx_fact_movie   ON fact_ratings(movie_id);
CREATE INDEX idx_fact_time    ON fact_ratings(time_id);
CREATE INDEX idx_fact_rating  ON fact_ratings(rating);
"""

with engine.connect() as conn:
    conn.execute(text(schema_sql))
    conn.commit()

print("✅ Star schema created successfully")
print("""
Tables created:
  dim_movies          ← movie dimension
  dim_users           ← user dimension
  dim_time            ← time dimension
  dim_genres          ← genre dimension
  bridge_movie_genre  ← movie↔genre bridge
  fact_ratings        ← central fact table
""")

✅ Star schema created successfully

Tables created:
  dim_movies          ← movie dimension
  dim_users           ← user dimension
  dim_time            ← time dimension
  dim_genres          ← genre dimension
  bridge_movie_genre  ← movie↔genre bridge
  fact_ratings        ← central fact table



In [4]:
# Load Dimension:Movies

import ast

movies    = pd.read_csv(PROC + 'movies_master.csv')
mov_stats = pd.read_csv(FEAT + 'movie_interaction_features.csv')
temporal  = pd.read_csv(FEAT + 'temporal_features.csv')

def safe_parse_list(val):
    try:
        result = ast.literal_eval(str(val))
        return result if isinstance(result, list) else []
    except:
        return []

movies['cast_names']   = movies['cast_names'].apply(safe_parse_list)
movies['genres_list']  = movies['genres_list'].apply(safe_parse_list)
movies['keyword_list'] = movies['keyword_list'].apply(safe_parse_list)

# Merge with stats and temporal
movies = movies.merge(
    mov_stats[['movieId', 'rating_count',
               'rating_mean', 'popularity_tier']],
    on='movieId', how='left'
)
movies = movies.merge(
    temporal[['id', 'era']],
    on='id', how='left'
)

# Only load movies that have a movieId
dim_movies = movies[movies['movieId'].notna()].copy()
dim_movies['movieId'] = dim_movies['movieId'].astype(int)

dim_movies_load = pd.DataFrame({
    'movie_id':          dim_movies['movieId'],
    'tmdb_id':           dim_movies['id'],
    'imdb_id':           dim_movies['imdb_id'].fillna(''),
    'title':             dim_movies['title'].fillna(''),
    'original_title':    dim_movies['original_title'].fillna(''),
    'overview':          dim_movies['overview'].fillna(''),
    'tagline':           dim_movies['tagline'].fillna(''),
    'original_language': dim_movies['original_language'].fillna(''),
    'release_year':      pd.to_numeric(
                             dim_movies['year'],
                             errors='coerce'),
    'era':               dim_movies['era'].fillna('unknown'),
    'runtime':           dim_movies['runtime'],
    'budget':            dim_movies['budget'],
    'revenue':           dim_movies['revenue'],
    'vote_average':      dim_movies['vote_average'],
    'vote_count':        dim_movies['vote_count'],
    'popularity':        dim_movies['popularity'],
    'director':          dim_movies['director'].fillna(''),
    'top_cast':          dim_movies['cast_names'].apply(
                             lambda x: ', '.join(x[:3]) if x else ''),
    'poster_path':       dim_movies['poster_path'].fillna(''),
    'rating_count':      dim_movies['rating_count'].fillna(0).astype(int),
    'rating_mean':       dim_movies['rating_mean'].fillna(0),
    'popularity_tier':   dim_movies['popularity_tier'].fillna('cold'),
})

# Remove duplicates on movie_id
dim_movies_load = dim_movies_load.drop_duplicates(
    subset='movie_id')

dim_movies_load.to_sql(
    'dim_movies', engine,
    if_exists='append', index=False, method='multi',
    chunksize=500
)

print(f"✅ dim_movies loaded: {len(dim_movies_load):,} rows")

✅ dim_movies loaded: 45,454 rows


In [5]:
## Load Dimension:Users

user_features = pd.read_csv(FEAT + 'user_features.csv')

dim_users_load = pd.DataFrame({
    'user_id':          user_features['userId'],
    'rating_count':     user_features['rating_count'],
    'rating_mean':      user_features['rating_mean'].round(3),
    'rating_std':       user_features['rating_std'].round(3),
    'unique_movies':    user_features['unique_movies'],
    'activity_tier':    user_features['activity_tier'],
    'rating_tendency':  user_features['rating_tendency'],
    'top_genre':        user_features['top_genre'].fillna('Unknown'),
})

dim_users_load.to_sql(
    'dim_users', engine,
    if_exists='append', index=False, method='multi',
    chunksize=500
)

print(f"✅ dim_users loaded: {len(dim_users_load):,} rows")

✅ dim_users loaded: 671 rows


In [ ]:
## Load Dimension:Time

ratings = pd.read_csv(PROC + 'ratings_cleaned.csv')
ratings['timestamp'] = pd.to_datetime(
    ratings['timestamp'], errors='coerce')

# Build time dimension from unique timestamps
time_dim = ratings[['timestamp']].drop_duplicates().copy()
time_dim = time_dim.dropna()
time_dim['year']        = time_dim['timestamp'].dt.year
time_dim['month']       = time_dim['timestamp'].dt.month
time_dim['day']         = time_dim['timestamp'].dt.day
time_dim['hour']        = time_dim['timestamp'].dt.hour
time_dim['day_of_week'] = time_dim['timestamp'].dt.dayofweek
time_dim['quarter']     = time_dim['timestamp'].dt.quarter

time_dim.to_sql(
    'dim_time', engine,
    if_exists='append', index=False, method='multi',
    chunksize=1000
)

print(f"✅ dim_time loaded: {len(time_dim):,} rows")